# 04 NDJF Catalog

This notebook builds the first-pass November-December-January-February ERA5 JPCZ catalog using the stabilized module workflow. It reuses completed December checkpoints when available, applies month-specific divergence thresholds, and writes the final master catalog to Google Drive.

In [3]:
CATALOG_YEARS = tuple(range(2000, 2019))
CATALOG_MONTHS = (11, 12, 1, 2)
WRITE_FINAL_OUTPUTS = True

print("CATALOG_YEARS:", CATALOG_YEARS)
print("CATALOG_MONTHS:", CATALOG_MONTHS)
print("WRITE_FINAL_OUTPUTS:", WRITE_FINAL_OUTPUTS)


Mounted at /content/drive
Persistent output dir: /content/drive/MyDrive/JPCZcatalog_outputs

Cloning into '/content/JPCZcatalog'...

Working directory: /content/JPCZcatalog


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = True
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [4]:
from pathlib import Path
import pandas as pd
import xarray as xr

from jpcz_catalog.detect import count_events_from_threshold, threshold_from_std

CATALOG_YEARS = tuple(range(2000, 2019))
CATALOG_MONTHS = (11, 12, 1, 2)

drive_dir = Path("/content/drive/MyDrive/JPCZcatalog_outputs")
ndjf_checkpoint_dir = drive_dir / "ndjf_monthly"
december_checkpoint_dir = drive_dir / "december_yearly"

if not ndjf_checkpoint_dir.exists():
    raise FileNotFoundError(f"Missing NDJF checkpoint directory: {ndjf_checkpoint_dir}")

monthly_records = []

for year in CATALOG_YEARS:
    for month in CATALOG_MONTHS:
        reusable_december_path = december_checkpoint_dir / f"december_{year}_D.nc"
        checkpoint_path = (
            reusable_december_path
            if month == 12 and reusable_december_path.exists()
            else ndjf_checkpoint_dir / f"{year}_{month:02d}_D.nc"
        )

        if not checkpoint_path.exists():
            raise FileNotFoundError(f"Missing checkpoint file: {checkpoint_path}")

        D_valid = xr.open_dataarray(checkpoint_path).load()
        monthly_records.append({
            "year": year,
            "month": month,
            "D": D_valid,
        })

        print(
            f"Loaded {year}-{month:02d}: "
            f"valid_D={D_valid.sizes['time']}, "
            f"peak_D={float(D_valid.min().values):.3e} s^-1"
        )

month_threshold_rows = []
threshold_by_month = {}

for month in CATALOG_MONTHS:
    month_D = xr.concat(
        [record["D"] for record in monthly_records if record["month"] == month],
        dim="time",
    ).sortby("time")

    D_mean, D_std, D_threshold = threshold_from_std(month_D, n_std=2.0)
    threshold_by_month[month] = D_threshold
    month_threshold_rows.append({
        "month": month,
        "D_mean_s-1": D_mean,
        "D_std_s-1": D_std,
        "threshold_s-1": D_threshold,
        "threshold_1e5_s-1": D_threshold * 1e5,
    })

month_threshold_df = pd.DataFrame(month_threshold_rows).sort_values("month")

event_frames = []
for record in monthly_records:
    events_month = count_events_from_threshold(record["D"], threshold_by_month[record["month"]]).copy()
    if events_month.empty:
        continue

    events_month["year"] = record["year"]
    events_month["month"] = record["month"]
    events_month["season_year"] = record["year"] if record["month"] in (1, 2) else record["year"] + 1
    events_month["detection_threshold_s-1"] = threshold_by_month[record["month"]]
    events_month["detection_threshold_1e5_s-1"] = threshold_by_month[record["month"]] * 1e5
    event_frames.append(events_month)

ndjf_events_df = pd.concat(event_frames, ignore_index=True).sort_values("event_peak").reset_index(drop=True)

print("Recovered NDJF events:", len(ndjf_events_df))
print(month_threshold_df)


Loaded 2000-11: valid_D=709, peak_D=-2.067e-05 s^-1
Loaded 2000-12: valid_D=733, peak_D=-3.415e-05 s^-1
Loaded 2000-01: valid_D=733, peak_D=-2.972e-05 s^-1
Loaded 2000-02: valid_D=685, peak_D=-4.312e-05 s^-1
Loaded 2001-11: valid_D=709, peak_D=-2.191e-05 s^-1
Loaded 2001-12: valid_D=733, peak_D=-3.852e-05 s^-1
Loaded 2001-01: valid_D=733, peak_D=-3.137e-05 s^-1
Loaded 2001-02: valid_D=661, peak_D=-3.535e-05 s^-1
Loaded 2002-11: valid_D=709, peak_D=-3.740e-05 s^-1
Loaded 2002-12: valid_D=733, peak_D=-3.087e-05 s^-1
Loaded 2002-01: valid_D=733, peak_D=-3.926e-05 s^-1
Loaded 2002-02: valid_D=661, peak_D=-3.217e-05 s^-1
Loaded 2003-11: valid_D=709, peak_D=-1.810e-05 s^-1
Loaded 2003-12: valid_D=733, peak_D=-3.910e-05 s^-1
Loaded 2003-01: valid_D=733, peak_D=-3.335e-05 s^-1
Loaded 2003-02: valid_D=661, peak_D=-2.032e-05 s^-1
Loaded 2004-11: valid_D=709, peak_D=-3.323e-05 s^-1
Loaded 2004-12: valid_D=733, peak_D=-2.981e-05 s^-1
Loaded 2004-01: valid_D=733, peak_D=-3.843e-05 s^-1
Loaded 2004-

In [5]:
from pathlib import Path
import shutil

local_dir = Path("outputs/verification")
local_dir.mkdir(parents=True, exist_ok=True)

ndjf_events_path = local_dir / "ndjf_events_first_pass.csv"
month_thresholds_path = local_dir / "ndjf_monthly_thresholds.csv"

ndjf_events_df.to_csv(ndjf_events_path, index=False)
month_threshold_df.to_csv(month_thresholds_path, index=False)

drive_dir = Path("/content/drive/MyDrive/JPCZcatalog_outputs")
for path in [ndjf_events_path, month_thresholds_path]:
    shutil.copy2(path, drive_dir / path.name)
    print(f"Copied {path} -> {drive_dir / path.name}")


Copied outputs/verification/ndjf_events_first_pass.csv -> /content/drive/MyDrive/JPCZcatalog_outputs/ndjf_events_first_pass.csv
Copied outputs/verification/ndjf_monthly_thresholds.csv -> /content/drive/MyDrive/JPCZcatalog_outputs/ndjf_monthly_thresholds.csv


In [6]:
from pathlib import Path
import shutil

import pandas as pd
import xarray as xr

from jpcz_catalog.config import DECEMBER_BENCHMARK_YEARS, JPCZ_POLYGON_VERTICES
from jpcz_catalog.detect import (
    compute_polygon_mean_divergence_series,
    count_events_from_threshold,
    prepare_detection_geometry,
    threshold_from_std,
)
from jpcz_catalog.era5 import open_arco_era5, open_month
from jpcz_catalog.verification import write_text_summary

ds = open_arco_era5()
print("valid_time_start:", ds.attrs.get("valid_time_start"))
print("valid_time_stop:", ds.attrs.get("valid_time_stop"))

geometry_ds = open_month(ds, CATALOG_YEARS[0], 11)
geometry = prepare_detection_geometry(
    geometry_ds.longitude,
    geometry_ds.latitude,
    JPCZ_POLYGON_VERTICES,
)

ndjf_checkpoint_dir = Path(DRIVE_OUTPUT_DIR) / "ndjf_monthly" if PERSIST_OUTPUTS_TO_DRIVE else Path("outputs/verification/ndjf_monthly")
december_checkpoint_dir = Path(DRIVE_OUTPUT_DIR) / "december_yearly"
ndjf_checkpoint_dir.mkdir(parents=True, exist_ok=True)
print("NDJF checkpoint dir:", ndjf_checkpoint_dir)

total_hourly_points = 0
monthly_records = []

for year in CATALOG_YEARS:
    for month in CATALOG_MONTHS:
        reusable_december_path = december_checkpoint_dir / f"december_{year}_D.nc"
        checkpoint_path = reusable_december_path if month == 12 and reusable_december_path.exists() else ndjf_checkpoint_dir / f"{year}_{month:02d}_D.nc"

        if checkpoint_path.exists():
            D_valid = xr.open_dataarray(checkpoint_path).load()
            hourly_points = int(D_valid.sizes["time"] + 11)
            total_hourly_points += hourly_points
            monthly_records.append({
                "year": year,
                "month": month,
                "hourly_points": hourly_points,
                "D": D_valid,
            })
            print(
                f"{year}-{month:02d}: loaded checkpoint hourly={hourly_points}, "
                f"valid_D={D_valid.sizes['time']}, "
                f"peak_D={float(D_valid.min().values):.3e} s^-1"
            )
            continue

        month_ds = open_month(ds, year, month)
        hourly_month, D_month, _ = compute_polygon_mean_divergence_series(
            month_ds,
            geometry=geometry,
        )
        D_valid = D_month.dropna("time")
        D_valid.to_netcdf(checkpoint_path)

        total_hourly_points += int(hourly_month.sizes["time"])
        monthly_records.append({
            "year": year,
            "month": month,
            "hourly_points": int(hourly_month.sizes["time"]),
            "D": D_valid,
        })
        print(
            f"{year}-{month:02d}: hourly={hourly_month.sizes['time']}, "
            f"valid_D={D_valid.sizes['time']}, "
            f"peak_D={float(D_valid.min().values):.3e} s^-1"
        )

month_threshold_rows = []
threshold_by_month = {}

for month in CATALOG_MONTHS:
    month_D = xr.concat([record["D"] for record in monthly_records if record["month"] == month], dim="time").sortby("time")
    D_mean, D_std, D_threshold = threshold_from_std(month_D, n_std=2.0)
    threshold_by_month[month] = D_threshold
    month_threshold_rows.append({
        "month": month,
        "D_mean_s-1": D_mean,
        "D_std_s-1": D_std,
        "threshold_s-1": D_threshold,
        "threshold_1e5_s-1": D_threshold * 1e5,
    })

month_threshold_df = pd.DataFrame(month_threshold_rows).sort_values("month")

event_frames = []
for record in monthly_records:
    events_month = count_events_from_threshold(record["D"], threshold_by_month[record["month"]]).copy()
    if events_month.empty:
        continue

    events_month["year"] = record["year"]
    events_month["month"] = record["month"]
    events_month["season_year"] = record["year"] if record["month"] in (1, 2) else record["year"] + 1
    events_month["detection_threshold_s-1"] = threshold_by_month[record["month"]]
    events_month["detection_threshold_1e5_s-1"] = threshold_by_month[record["month"]] * 1e5
    event_frames.append(events_month)

if event_frames:
    ndjf_events_df = pd.concat(event_frames, ignore_index=True).sort_values("event_peak").reset_index(drop=True)
else:
    ndjf_events_df = pd.DataFrame()

if WRITE_FINAL_OUTPUTS:
    ndjf_events_path = Path("outputs/verification/ndjf_events_first_pass.csv")
    ndjf_events_df.to_csv(ndjf_events_path, index=False)
    month_thresholds_path = Path("outputs/verification/ndjf_monthly_thresholds.csv")
    month_threshold_df.to_csv(month_thresholds_path, index=False)

    if PERSIST_OUTPUTS_TO_DRIVE:
        for path in [ndjf_events_path, month_thresholds_path]:
            shutil.copy2(path, Path(DRIVE_OUTPUT_DIR) / path.name)
else:
    print("Checkpoint-only mode: skipping top-level NDJF first-pass outputs for this chunk run.")

print("Total hourly NDJF points:", total_hourly_points)
print("Total NDJF events:", len(ndjf_events_df))
print(month_threshold_df)
ndjf_events_df.head()


valid_time_start: 1940-01-01
valid_time_stop: 2025-12-31
NDJF checkpoint dir: /content/drive/MyDrive/JPCZcatalog_outputs/ndjf_monthly
2000-11: loaded checkpoint hourly=720, valid_D=709, peak_D=-2.067e-05 s^-1
2000-12: loaded checkpoint hourly=744, valid_D=733, peak_D=-3.415e-05 s^-1
2000-01: loaded checkpoint hourly=744, valid_D=733, peak_D=-2.972e-05 s^-1
2000-02: loaded checkpoint hourly=696, valid_D=685, peak_D=-4.312e-05 s^-1
2001-11: loaded checkpoint hourly=720, valid_D=709, peak_D=-2.191e-05 s^-1
2001-12: loaded checkpoint hourly=744, valid_D=733, peak_D=-3.852e-05 s^-1
2001-01: loaded checkpoint hourly=744, valid_D=733, peak_D=-3.137e-05 s^-1
2001-02: loaded checkpoint hourly=672, valid_D=661, peak_D=-3.535e-05 s^-1
2002-11: loaded checkpoint hourly=720, valid_D=709, peak_D=-3.740e-05 s^-1
2002-12: loaded checkpoint hourly=744, valid_D=733, peak_D=-3.087e-05 s^-1
2002-01: loaded checkpoint hourly=744, valid_D=733, peak_D=-3.926e-05 s^-1
2002-02: loaded checkpoint hourly=672, va

,event_start,event_end,event_peak,event_peak_D_s-1,event_peak_D_1e5_s-1,duration_hours,year,month,season_year,detection_threshold_s-1,detection_threshold_1e5_s-1
0,2000-01-05 23:00:00,2000-01-06 00:00:00,2000-01-05 23:00:00,-0.000024,-2.438026,2,2000,1,2000,-0.000024,-2.425193
1,2000-01-13 08:00:00,2000-01-13 14:00:00,2000-01-13 11:00:00,-0.000030,-2.972377,7,2000,1,2000,-0.000024,-2.425193
2,2000-01-19 22:00:00,2000-01-20 08:00:00,2000-01-20 04:00:00,-0.000029,-2.927972,11,2000,1,2000,-0.000024,-2.425193
3,2000-02-07 13:00:00,2000-02-08 13:00:00,2000-02-08 00:00:00,-0.000043,-4.312036,25,2000,2,2000,-0.000020,-1.957567
4,2000-02-13 18:00:00,2000-02-14 03:00:00,2000-02-13 22:00:00,-0.000024,-2.398227,10,2000,2,2000,-0.000020,-1.957567


In [7]:
from pathlib import Path
import shutil

import numpy as np
import pandas as pd

from jpcz_catalog.classify import (
    annotate_events_with_environment,
    compute_month_climatological_slp_index,
)
from jpcz_catalog.verification import render_ndjf_catalog_summary, write_text_summary

if not WRITE_FINAL_OUTPUTS:
    raise RuntimeError("Set WRITE_FINAL_OUTPUTS = True before running the final NDJF classification cell.")

if "ndjf_events_df" not in globals():
    ndjf_events_df = pd.read_csv(
        Path("outputs/verification/ndjf_events_first_pass.csv"),
        parse_dates=["event_start", "event_end", "event_peak"],
    )

ndjf_catalog = annotate_events_with_environment(ds, ndjf_events_df)

slp_mean_by_month = {}
slp_std_by_month = {}
for month in CATALOG_MONTHS:
    _, mean_value, std_value = compute_month_climatological_slp_index(ds, month, years=CATALOG_YEARS)
    slp_mean_by_month[month] = mean_value
    slp_std_by_month[month] = std_value

ndjf_catalog["slp_climatology_mean_hpa"] = ndjf_catalog["month"].map(slp_mean_by_month)
ndjf_catalog["slp_climatology_std_hpa"] = ndjf_catalog["month"].map(slp_std_by_month)
ndjf_catalog["monsoon_type"] = np.where(
    ndjf_catalog["slp_diff_hpa"] > ndjf_catalog["slp_climatology_mean_hpa"],
    "Type 1 strong-monsoon",
    "Type 2 weak-monsoon",
)

type1_mask = ndjf_catalog["monsoon_type"] == "Type 1 strong-monsoon"
if type1_mask.sum() == 0:
    raise ValueError("No Type 1 strong-monsoon events were detected in the NDJF catalog.")

zeta_split_ndjf = (
    ndjf_catalog.loc[type1_mask, "zeta_box_mean_s-1"].mean()
    + ndjf_catalog.loc[type1_mask, "zeta_box_mean_s-1"].std()
)

ndjf_catalog["shinoda_class"] = "Type 2 weak-monsoon"
ndjf_catalog.loc[
    type1_mask & (ndjf_catalog["zeta_box_mean_s-1"] > zeta_split_ndjf),
    "shinoda_class",
] = "Type 1B higher-vorticity"
ndjf_catalog.loc[
    type1_mask & (ndjf_catalog["zeta_box_mean_s-1"] <= zeta_split_ndjf),
    "shinoda_class",
] = "Type 1A lower-vorticity"

slp_thresholds_df = pd.DataFrame(
    {
        "month": list(CATALOG_MONTHS),
        "slp_climatology_mean_hpa": [slp_mean_by_month[m] for m in CATALOG_MONTHS],
        "slp_climatology_std_hpa": [slp_std_by_month[m] for m in CATALOG_MONTHS],
    }
).sort_values("month")

slp_thresholds_path = Path("outputs/verification/ndjf_monsoon_thresholds.csv")
slp_thresholds_df.to_csv(slp_thresholds_path, index=False)

final_catalog_path = Path("outputs/verification/jpcz_catalog_ndjf.csv")
ndjf_catalog.to_csv(final_catalog_path, index=False)

summary_text = render_ndjf_catalog_summary(
    total_events=len(ndjf_catalog),
    month_threshold_df=month_threshold_df,
    catalog_df=ndjf_catalog,
)
summary_path = write_text_summary("outputs/verification/jpcz_catalog_ndjf_summary.md", summary_text)

if PERSIST_OUTPUTS_TO_DRIVE:
    for path in [slp_thresholds_path, final_catalog_path, summary_path]:
        shutil.copy2(path, Path(DRIVE_OUTPUT_DIR) / path.name)

print(summary_text)
print("NDJF zeta split:", zeta_split_ndjf, "s^-1")
ndjf_catalog.head()


# NDJF Catalog Summary

- Total NDJF events: 210

## Monthly thresholds
 month  D_mean_s-1  D_std_s-1  threshold_s-1  threshold_1e5_s-1
     1   -0.000004   0.000010      -0.000024          -2.425193
     2   -0.000002   0.000009      -0.000020          -1.957567
    11   -0.000002   0.000008      -0.000018          -1.821653
    12   -0.000005   0.000010      -0.000026          -2.569157

## Event counts by month
month
1     55
2     49
11    60
12    46

## Monsoon types
monsoon_type
Type 1 strong-monsoon    127
Type 2 weak-monsoon       83

## Final classes
shinoda_class
Type 1A lower-vorticity     104
Type 2 weak-monsoon          83
Type 1B higher-vorticity     23

Interpretation:
This NDJF catalog extends the Shinoda-style December detector to November, December, January, and February using the same core divergence logic with month-specific climatological thresholds.
NDJF zeta split: 1.590559282791226e-05 s^-1


,event_start,event_end,event_peak,event_peak_D_s-1,event_peak_D_1e5_s-1,duration_hours,year,month,season_year,detection_threshold_s-1,detection_threshold_1e5_s-1,slp_diff_hpa,zeta_box_mean_s-1,slp_climatology_mean_hpa,slp_climatology_std_hpa,monsoon_type,shinoda_class
0,2000-01-05 23:00:00,2000-01-06 00:00:00,2000-01-05 23:00:00,-0.000024,-2.438026,2,2000,1,2000,-0.000024,-2.425193,-15.108672,0.000005,11.127163,8.703955,Type 2 weak-monsoon,Type 2 weak-monsoon
1,2000-01-13 08:00:00,2000-01-13 14:00:00,2000-01-13 11:00:00,-0.000030,-2.972377,7,2000,1,2000,-0.000024,-2.425193,-0.301875,0.000007,11.127163,8.703955,Type 2 weak-monsoon,Type 2 weak-monsoon
2,2000-01-19 22:00:00,2000-01-20 08:00:00,2000-01-20 04:00:00,-0.000029,-2.927972,11,2000,1,2000,-0.000024,-2.425193,21.598672,0.000016,11.127163,8.703955,Type 1 strong-monsoon,Type 1B higher-vorticity
3,2000-02-07 13:00:00,2000-02-08 13:00:00,2000-02-08 00:00:00,-0.000043,-4.312036,25,2000,2,2000,-0.000020,-1.957567,8.146719,0.000014,8.544207,8.269666,Type 2 weak-monsoon,Type 2 weak-monsoon
4,2000-02-13 18:00:00,2000-02-14 03:00:00,2000-02-13 22:00:00,-0.000024,-2.398227,10,2000,2,2000,-0.000020,-1.957567,3.847500,-0.000003,8.544207,8.269666,Type 2 weak-monsoon,Type 2 weak-monsoon
